# 03 — Embeddings y recuperación: ver el R de RAG

En el notebook anterior convertimos seis fuentes en 17 fragmentos identificables. Ahora transformaremos textos en vectores y buscaremos los más próximos a una pregunta. **No generamos una respuesta**: la salida será una lista ordenada de evidencias candidatas.

El recorrido es: chunks → textos para búsqueda → embeddings → vector de la pregunta → similitud → top-k → inspección de las fuentes.

Trabajamos con un modelo de embeddings local en CPU. La primera ejecución descarga sus archivos desde Hugging Face; luego se reutilizan desde la caché del proyecto. El texto del corpus no se envía al servicio de descarga. No se utiliza `.env`, la API de OpenAI ni crédito de API. La descarga y el cálculo local sí requieren tiempo, disco y memoria.

Ejecutar con **Python (rag-demo)**, de arriba abajo. Shift+Enter ejecuta la celda seleccionada. Si reiniciás el kernel, las variables se pierden y debés comenzar de nuevo. No hace falta ejecutar los notebooks anteriores si `data/chunks.jsonl` está actualizado.

Al terminar deberías poder explicar qué es una fila de la matriz, dónde se calcula la similitud y por qué el primer resultado no es necesariamente una fuente autorizada para responder.

## 1. Leer los chunks y comprobar su procedencia

`Path` permite ubicar el proyecto aunque Jupyter haya iniciado en `notebooks/`. Buscamos hacia arriba, igual que antes. `json.loads` convierte cada línea del JSONL en un diccionario; la lista resultante mantiene su orden.

La correspondencia **fila i ↔ chunk i** será esencial: si ordenamos una matriz sin mantener los IDs, podríamos citar la fuente equivocada. Primero verificamos IDs únicos, número de chunks y que el corpus no haya cambiado desde la exportación. SHA-256 compara bytes, no significados: un cambio de formato también puede requerir regenerar datos.

No leemos respuestas esperadas para construir embeddings. Las preguntas se cargarán más adelante como consultas, sin indexarlas. Si falta un archivo o aparece un error de vigencia de los datos, ejecutar nuevamente el notebook 02.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import numpy as np
from IPython.display import display, Markdown

actual = Path.cwd().resolve()
raiz = next((p for p in [actual, *actual.parents]
             if (p / "requirements.in").is_file() and (p / "corpus").is_dir()), None)
if raiz is None:
    raise RuntimeError("Abrí el notebook dentro de rag-demo.")
ruta_chunks = raiz / "data/chunks.jsonl"
chunks = [json.loads(linea) for linea in ruta_chunks.read_text(encoding="utf-8").splitlines()]
manifiesto = json.loads((raiz / "data/manifest.json").read_text(encoding="utf-8"))
if not chunks or len(chunks) != manifiesto["chunks"]:
    raise ValueError("Exportación incompleta: ejecutar notebook 02.")
if len({c["chunk_id"] for c in chunks}) != len(chunks):
    raise ValueError("IDs de chunk duplicados.")
fuentes = {s["source"]: s["sha256"] for s in manifiesto["sources"]}
actuales = {p.relative_to(raiz).as_posix() for p in (raiz / "corpus").glob("*.md")}
if actuales != set(fuentes):
    raise ValueError("Cambió el conjunto de fuentes; ejecutar notebook 02.")
for fuente, huella in fuentes.items():
    if hashlib.sha256((raiz / fuente).read_bytes()).hexdigest() != huella:
        raise ValueError(f"Cambió {fuente}; ejecutar notebook 02.")
for c in chunks:
    if c["source_sha256"] != fuentes[c["source"]]:
        raise ValueError("El chunk y el manifiesto no corresponden a la misma fuente.")
print("Chunks cargados:", len(chunks))
print("Primer ID:", chunks[0]["chunk_id"])

**Comprobar la salida:** deberían cargarse 17 chunks. Si el chequeo de fuentes falla, no continuar con una matriz anterior: regenerar primero el notebook 02.

## 2. Cargar el modelo de embeddings

Un embedding es un vector de números aprendido por un modelo para representar relaciones entre textos. No entrenamos el modelo con nuestras políticas: utilizamos uno preentrenado para codificarlas. Eso es distinto de ajustar sus pesos con fine-tuning.

Elegimos **intfloat/multilingual-e5-small**, que admite español. La revisión es un identificador fijo del repositorio, para evitar que una actualización silenciosa altere la demo. `SentenceTransformer` reúne tokenizador y modelo. `device="cpu"` permite usarlo sin configurar una GPU; `trust_remote_code=False` evita cargar código personalizado del repositorio del modelo.

La caché queda en `.cache/huggingface/`, excluida de Git. La primera carga necesita internet; las siguientes buscan primero esa revisión en los archivos locales. Si faltan, se solicita la descarga. No confundas esta caché con la matriz de embeddings: contiene el modelo, no los vectores de nuestros documentos.

Las variables de entorno se fijan antes de importar las bibliotecas. La telemetría y el uso implícito de credenciales de Hugging Face quedan desactivados; el modelo es público. Limitamos PyTorch a cuatro hilos como máximo para que la demo no use todos los núcleos disponibles.

In [ ]:
os.environ["HF_HOME"] = str(raiz / ".cache/huggingface")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
from sentence_transformers import SentenceTransformer
import torch

torch.set_num_threads(min(4, os.cpu_count() or 1))
MODELO = "intfloat/multilingual-e5-small"
REVISION = "614241f622f53c4eeff9890bdc4f31cfecc418b3"
parametros = dict(device="cpu", revision=REVISION,
                  cache_folder=str(raiz / ".cache/huggingface/hub"),
                  trust_remote_code=False, token=False)
try:
    modelo = SentenceTransformer(MODELO, local_files_only=True, **parametros)
except OSError:
    print("Primera carga: descargando el modelo público...")
    modelo = SentenceTransformer(MODELO, **parametros)
print("Modelo:", MODELO)
print("Dimensiones:", modelo.get_embedding_dimension())
print("Límite de tokens por entrada:", modelo.max_seq_length)

**Comprobar la salida:** el modelo tiene 384 dimensiones y un límite de 512 tokens por entrada. La primera descarga puede tardar; las advertencias de caché sobre symlinks en Windows no exigen permisos de administrador.

## 3. Elegir qué texto representar y medirlo

En el paso anterior guardamos `retrieval_text` con todos los metadatos serializados. Aquí hacemos una decisión explícita: para calcular similitud usaremos una versión textual más compacta, sin claves técnicas como `document_id`, `source_sha256` o etiquetas YAML. Conservamos título, audiencia, introducción, contexto adicional y sección. Los metadatos completos siguen disponibles en el chunk y se mostrarán al inspeccionar resultados.

Esto evita que las etiquetas administrativas ocupen gran parte de la entrada del modelo. Es una hipótesis de diseño, no una garantía de mejor ranking. La evidencia que mostraremos sigue siendo el fragmento completo del notebook 02; no reemplazamos ese archivo.

E5 requiere los prefijos **`passage: `** para documentos y **`query: `** para consultas, incluso en español. Son convenciones de entrenamiento del modelo, no instrucciones para un asistente.

`tokenizer(..., truncation=False)` mide la entrada completa incluyendo tokens especiales. El conteo depende del tokenizador: no equivale a palabras ni al conteo de la API generativa. Si excedemos el límite, detenemos el flujo para revisar el chunking; no aceptamos truncamiento silencioso que pudiera borrar una condición al final.

In [ ]:
def texto_para_embedding(chunk):
    partes = [chunk["metadata"]["titulo"],
              "Audiencia: " + chunk["metadata"]["audiencia"],
              chunk["introduction"]]
    partes += [s["title"] + "\n" + s["text"] for s in chunk["context_sections"]]
    partes += [chunk["section_title"], chunk["text"]]
    return "\n\n".join(p for p in partes if p)

textos = ["passage: " + texto_para_embedding(c) for c in chunks]
longitudes = [len(modelo.tokenizer(t, truncation=False)["input_ids"]) for t in textos]
for c, cantidad in zip(chunks, longitudes):
    print(c["chunk_id"], "→", cantidad, "tokens")
largos = [(c["chunk_id"], n) for c, n in zip(chunks, longitudes) if n > modelo.max_seq_length]
if largos:
    raise ValueError(f"Hay fragmentos demasiado largos; revisar división antes de continuar: {largos}")
print("\nTexto exacto del primer embedding:\n", textos[0])

**Comprobar la salida:** en la ejecución de preparación el máximo fue 271 tokens, por debajo del límite. Este valor se vuelve a medir: no asumir que seguirá igual si se modifica el corpus.

## 4. Codificar documentos: de 17 textos a una matriz

`encode` aplica el mismo modelo a cada texto y devuelve sus vectores en el mismo orden. `batch_size=4` procesa grupos pequeños para moderar memoria; no cambia la cantidad de chunks. `convert_to_numpy=True` devuelve una matriz de NumPy fácil de inspeccionar.

Con `normalize_embeddings=True`, cada vector tiene norma euclídea aproximadamente igual a 1. Gracias a esa normalización, después podremos calcular la similitud coseno con un producto matricial. Normalizar no elimina errores semánticos ni convierte los números en probabilidades.

`shape` indica filas y columnas. Cada fila corresponde a un chunk; cada columna es una dimensión aprendida. No podemos interpretar una coordenada aislada como «monto», «vigencia» o «Argentina». Mostrar ocho números sirve para ver la representación, no para atribuirle significado a cada posición.

In [ ]:
embeddings = modelo.encode(textos, batch_size=4, convert_to_numpy=True,
                           normalize_embeddings=True, show_progress_bar=True)
assert embeddings.shape == (len(chunks), modelo.get_embedding_dimension())
assert np.isfinite(embeddings).all(), "La matriz contiene valores inválidos."
assert np.allclose(np.linalg.norm(embeddings, axis=1), 1.0, atol=1e-5)
print("Forma de la matriz:", embeddings.shape)
print("Primeras ocho coordenadas del primer chunk:", embeddings[0, :8])
print("Tamaño de la matriz en memoria:", embeddings.nbytes, "bytes")

**Comprobar la salida:** la forma debe ser `(17, 384)`. Los números de cada fila representan un fragmento completo; no son una puntuación de calidad de ese fragmento.

## 5. Codificar una pregunta y calcular todas las similitudes

Leemos solamente las preguntas de la tabla Markdown. El patrón reconoce filas Q01…Q10 y extrae su texto; la rúbrica no interviene. Para la búsqueda usamos la pregunta, sin repetir el contexto administrativo común en cada consulta.

Elegimos Q01. La pregunta recibe el prefijo `query: ` y se representa con **el mismo modelo y normalización** que los documentos. Mezclar modelos produciría vectores que no se pueden comparar de forma significativa, incluso si tuvieran igual número de columnas.

La fórmula es coseno(q, d) = (q · d) / (||q|| ||d||). Como ambas normas son 1, basta q · d. `embeddings @ vector_consulta` multiplica una matriz de tamaño N×D por un vector D y produce N puntuaciones: una por chunk. El operador `@` hace producto matricial; `*` sería multiplicación elemento por elemento.

No existe aquí una base vectorial externa. Con 17 filas podemos comparar exhaustivamente contra todos los documentos. Una infraestructura de búsqueda aproximada será relevante a otra escala, pero no es necesaria para entender el mecanismo.

In [ ]:
tabla = (raiz / "eval/preguntas.md").read_text(encoding="utf-8")
preguntas = dict(re.findall(r"^\| (Q\d+) \| [^|]+ \| (.+?) \|$", tabla, re.M))
if len(preguntas) != 10:
    raise ValueError("No se pudieron leer los diez casos de preguntas.md.")

def codificar_consulta(pregunta):
    if not isinstance(pregunta, str) or not pregunta.strip():
        raise ValueError("La pregunta debe ser texto no vacío.")
    texto = "query: " + pregunta.strip()
    if len(modelo.tokenizer(texto, truncation=False)["input_ids"]) > modelo.max_seq_length:
        raise ValueError("La consulta supera el límite de tokens del modelo.")
    return modelo.encode(texto, convert_to_numpy=True, normalize_embeddings=True)

caso = "Q01"
pregunta = preguntas[caso]
vector_consulta = codificar_consulta(pregunta)
scores = embeddings @ vector_consulta
print(caso, pregunta)
print("Vector de consulta:", vector_consulta.shape)
print("Puntuaciones:", scores.shape)
for c, score in zip(chunks, scores):
    print(f"{score:.4f} | {c['chunk_id']}")

**Comprobar la salida:** hay 17 puntuaciones, todavía en el orden original de los chunks. El primer score impreso no es necesariamente el mayor.

## 6. Ordenar y seleccionar top-k

`np.argsort` devuelve las **posiciones** que ordenarían una lista, no los valores. Al ordenar `-scores`, los puntajes mayores quedan primero. `kind="stable"` mantiene el orden original cuando hay empates. `[:k]` toma las primeras k posiciones.

Con esas posiciones recuperamos los chunks originales: la asociación fila ↔ fuente queda preservada. `float(...)` e `int(...)` convierten escalares NumPy en tipos Python, útiles para serializar resultados después.

`k=3` significa «traer hasta tres candidatos», no «encontrar tres pruebas suficientes». La función rechaza valores inválidos de k y lo limita al número de chunks. Aumentar k puede agregar evidencia necesaria, pero también ruido, duplicados y reglas vencidas.

El score es afinidad bajo este modelo. **0,85 no significa 85 % de confianza**. No introducimos un umbral de abstención arbitrario: primero observamos resultados, luego será necesario evaluarlo con consultas representativas.

In [ ]:
def seleccionar_top_k(scores, k):
    if isinstance(k, bool) or not isinstance(k, int) or k < 1:
        raise ValueError("k debe ser un entero positivo.")
    indices = np.argsort(-scores, kind="stable")[:min(k, len(chunks))]
    return [{"rank": rango, "index": int(i), "score": float(scores[i]),
             "chunk": chunks[int(i)]} for rango, i in enumerate(indices, start=1)]

def recuperar(pregunta, k=3):
    vector = codificar_consulta(pregunta)
    return seleccionar_top_k(embeddings @ vector, k)

k = 3
resultados = seleccionar_top_k(scores, k)
for r in resultados:
    c = r["chunk"]
    print(f"{r['rank']}. {r['score']:.4f} | {c['chunk_id']} | {c['metadata']['estado']} | {c['metadata']['autoridad']}")

**Comprobar la salida:** ahora sí aparece el ranking. Revisar ID, estado y autoridad además del score. Un documento expirado puede quedar primero por afinidad textual.

## 7. Leer la evidencia: el ranking por sí solo no alcanza

Ahora mostramos el texto completo del chunk y su contexto, no una respuesta redactada. `display(Markdown(...))` sirve para separar visualmente resultados; `print` permite ver el contenido literal.

Antes de continuar a un LLM, preguntarnos: ¿aparece la regla necesaria? ¿Es vigente para el caso? ¿La audiencia coincide? ¿Hay una excepción que no recuperamos? ¿Se están contradiciendo fuentes de distinta autoridad?

No filtramos aún documentos expirados ni la FAQ. Mantenerlos permite mostrar que similitud, actualidad y autoridad son propiedades distintas. Para consultas históricas, incluso una política expirada puede ser relevante como evidencia del origen de una regla.

In [ ]:
for r in resultados:
    c = r["chunk"]
    display(Markdown(f"### {r['rank']}. {c['chunk_id']} — coseno {r['score']:.4f}"))
    print("Fuente:", c["source"])
    print(c["retrieval_text"])
    print("\n" + "─" * 60)

**Pausa antes de automatizar:** intentá responder usando solo lo que acaba de aparecer. Identificá qué dato o fuente falta y si hay conflictos que el score no resuelve.

## 8. Experimentar con preguntas y con k

Comparamos Q01 con su paráfrasis Q02, luego una excepción local (Q05), pérdida de tarjeta (Q07), ausencia de cobertura en el corpus (Q08) y datos insuficientes del usuario (Q09). Para cada pregunta calculamos una sola vez sus scores y observamos los prefijos de tamaño 1, 3 y 5.

La extracción del top-k **siempre devuelve candidatos**, incluso para odontología. El recuperador no concluye por sí mismo que exista un porcentaje documentado. En Q09 puede recuperar perfectamente la política y aun faltar la fecha de fin del viaje para decidir.

No llamamos a esto evaluación definitiva: no calculamos todavía métricas ni seleccionamos un modelo para ganar en estas diez consultas. Se trata de abrir la caja y formular hipótesis sobre errores. Probá cambiar un texto manualmente con `recuperar("tu pregunta", k=3)` sin modificar el corpus.

In [ ]:
experimentos = []
for caso_prueba in ["Q01", "Q02", "Q05", "Q07", "Q08", "Q09"]:
    consulta = preguntas[caso_prueba]
    puntajes = embeddings @ codificar_consulta(consulta)
    print("\n", caso_prueba, consulta)
    for cantidad in [1, 3, 5]:
        seleccion = seleccionar_top_k(puntajes, cantidad)
        print("k =", cantidad, "→", [(r["chunk"]["chunk_id"], round(r["score"], 4)) for r in seleccion])
        experimentos.append({"case_id": caso_prueba, "query": consulta, "k": cantidad,
                            "results": [{"chunk_id": r["chunk"]["chunk_id"], "score": r["score"]} for r in seleccion]})

**Lectura de la prueba del 2026-10-09:** Q01 ubicó la FAQ y la política expirada antes de la vigente; Q02 no incluyó la regla vigente en top-5. Q05 sí ubicó A2 primero. Son observaciones de esta versión y configuración, no resultados universales ni expectativas forzadas mediante asserts. Si cambiamos la estrategia, comparar de nuevo.

## 9. Verificar la matemática y guardar la matriz

Estas comprobaciones prueban propiedades del procedimiento, no calidad semántica. Calculamos el coseno de otra forma y lo comparamos con `@`; comprobamos orden descendente y que pedir más vecinos que chunks no duplique registros.

Guardamos `embeddings.npy`, el formato binario de NumPy, y un manifiesto JSON con IDs ordenados, modelo, revisión, prefijos y huella del archivo de chunks. **La matriz sola no es suficiente**: sin esos datos perderíamos su correspondencia con fuentes y su método de construcción. El archivo de experimentos sirve como registro de esta ejecución, no como verdad esperada.

`np.save` reemplaza la matriz anterior; no cambia el corpus. Al modificar corpus, modelo o texto de entrada, hay que regenerar embeddings. La igualdad después de `np.load(..., allow_pickle=False)` verifica la escritura y lectura sin usar serialización arbitraria de objetos.

In [ ]:
manual = np.array([np.dot(fila, vector_consulta) /
                   (np.linalg.norm(fila) * np.linalg.norm(vector_consulta)) for fila in embeddings])
assert np.allclose(scores, manual, atol=1e-6)
assert all(resultados[i]["score"] >= resultados[i + 1]["score"] for i in range(len(resultados) - 1))
assert len(seleccionar_top_k(scores, len(chunks) + 1)) == len(chunks)
assert len({r["chunk"]["chunk_id"] for r in seleccionar_top_k(scores, len(chunks))}) == len(chunks)

salida = raiz / "data"
np.save(salida / "embeddings.npy", embeddings, allow_pickle=False)
config_indice = {
    "schema_version": "1.0", "model": MODELO, "revision": REVISION,
    "normalized": True, "shape": list(embeddings.shape),
    "chunk_ids": [c["chunk_id"] for c in chunks],
    "chunks_sha256": hashlib.sha256(ruta_chunks.read_bytes()).hexdigest(),
    "text_strategy": "title_audience_introduction_context_sections_section_v1",
    "document_prefix": "passage: ", "query_prefix": "query: ",
    "embedding_texts": textos, "token_lengths": longitudes,
}
(salida / "embeddings_manifest.json").write_text(json.dumps(config_indice, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
(salida / "retrieval_examples.json").write_text(json.dumps(experimentos, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
assert np.array_equal(embeddings, np.load(salida / "embeddings.npy", allow_pickle=False))
print("Verificaciones correctas; matriz, manifiesto y ejemplos guardados en data/.")

**Comprobar la salida:** los tres archivos quedan en `data/`. Se verificó la matemática y la lectura de la matriz guardada, no la suficiencia semántica de todos los resultados. No usamos las respuestas esperadas para alterar el ranking.

## Qué logramos y qué falta

Ya podemos señalar físicamente dónde está el **R**: `embeddings @ vector` calcula afinidad y `seleccionar_top_k` elige candidatos. El modelo de embeddings transforma texto; no responde la pregunta ni decide qué política tiene autoridad.

Para discutir: ¿Q01 y Q02 recuperan la misma evidencia? ¿Q05 trae también el ámbito de la excepción? ¿Un k mayor mejora siempre la respuesta potencial? ¿Por qué Q08 obtiene vecinos aunque no tenga respuesta? ¿Qué cambiaría si solo codificáramos títulos?

El siguiente paso agregará la construcción del contexto y la llamada al LLM. Antes de eso, conviene poder justificar si los fragmentos disponibles alcanzan. Las futuras mejoras pueden incluir filtros, búsqueda léxica y evaluación separada de recuperación y generación.

Referencias de implementación: [SentenceTransformer](https://sbert.net/docs/package_reference/sentence_transformer/model.html) y [ficha del modelo E5](https://huggingface.co/intfloat/multilingual-e5-small). Los prefijos y el límite de entrada dependen del modelo: no asumir que todas las familias usan la misma convención.